# Notebook 04: Statistical Analysis
**Objective:** Apply correlation analysis, ANOVA, regression, and hypothesis testing to validate business insights.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/ecommerce_cleaned.csv')
print('Loaded:', df.shape)

In [ ]:
# ── 1. Correlation Analysis ──────────────────────────────
numeric_cols = ['price', 'quantity', 'total_amount', 'delivery_days', 'rating']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.3f', ax=ax)
ax.set_title('Correlation Matrix — Numeric Variables')
plt.tight_layout()
plt.show()

print('Key correlations:')
print(f'  delivery_days vs rating: {df["delivery_days"].corr(df["rating"]):.4f}')
print(f'  price vs rating: {df["price"].corr(df["rating"]):.4f}')
print(f'  total_amount vs rating: {df["total_amount"].corr(df["rating"]):.4f}')

In [ ]:
# ── 2. ANOVA: Rating across Categories ──────────────────
groups = [df[df['product_category']==c]['rating'].values 
          for c in df['product_category'].unique()]
f_stat, p_val = stats.f_oneway(*groups)
print(f'One-Way ANOVA (Rating ~ Category)')
print(f'  F-statistic: {f_stat:.4f}')
print(f'  p-value: {p_val:.4f}')
print(f'  Conclusion: {"Significant difference" if p_val < 0.05 else "No significant difference (p > 0.05)"}')

In [ ]:
# ── 3. ANOVA: Revenue across Cities ─────────────────────
city_groups = [df[df['city']==c]['total_amount'].values 
               for c in df['city'].unique()]
f2, p2 = stats.f_oneway(*city_groups)
print(f'One-Way ANOVA (Revenue ~ City)')
print(f'  F-statistic: {f2:.4f}, p-value: {p2:.4f}')
print(f'  Conclusion: {"Significant" if p2 < 0.05 else "Not significant"}')

In [ ]:
# ── 4. Chi-Square: Payment Method vs High-Value Orders ──
df_valid = df[df['payment_method'] != 'Unknown']
contingency = pd.crosstab(df_valid['payment_method'], df_valid['is_high_value'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f'Chi-Square: Payment Method vs High-Value Order')
print(f'  chi2={chi2:.4f}, p={p_chi:.4f}, dof={dof}')
print(f'  Conclusion: {"Association exists" if p_chi < 0.05 else "No significant association"}')

In [ ]:
# ── 5. Linear Regression: delivery_days -> rating ───────
from numpy.polynomial import polynomial as P
x = df['delivery_days'].values
y = df['rating'].values
slope, intercept, r_value, p_val_reg, std_err = stats.linregress(x, y)
print(f'Linear Regression: delivery_days → rating')
print(f'  Slope: {slope:.6f}')
print(f'  Intercept: {intercept:.4f}')
print(f'  R²: {r_value**2:.6f}')
print(f'  p-value: {p_val_reg:.4f}')
print(f'  Finding: Delivery speed explains near-zero variance in ratings')

In [ ]:
# ── 6. T-Test: Pune vs Other Cities Revenue ─────────────
pune_rev = df[df['city']=='Pune']['total_amount']
other_rev = df[df['city']!='Pune']['total_amount']
t_stat, t_p = stats.ttest_ind(pune_rev, other_rev)
print(f'T-Test: Pune revenue vs Other Cities')
print(f'  Pune mean: ₹{pune_rev.mean():,.0f}')
print(f'  Others mean: ₹{other_rev.mean():,.0f}')
print(f'  t={t_stat:.4f}, p={t_p:.4f}')
print(f'  Conclusion: {"Significant difference" if t_p < 0.05 else "No significant difference"}')

## Statistical Findings Summary
| Test | Variables | Result |
|------|-----------|--------|
| Correlation | Delivery vs Rating | r = -0.007 (negligible) |
| ANOVA | Rating across Categories | F=1.46, p=0.22 (not significant) |
| ANOVA | Revenue across Cities | Not significant |
| Chi-Square | Payment Method vs High-Value | No association |
| Regression | Delivery Days → Rating | R²≈0 (no predictive power) |
| T-Test | Pune vs Other Cities | Not statistically different |